<a href="https://colab.research.google.com/github/keyvanmahmoudi/Deep_Learning_Course/blob/main/solutions/Session07_convolutional_networks_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۷: شبکه‌های عصبی کانولوشنی (CNN)

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session07_convolutional_networks.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

> **نکتهٔ اجرا:** بعضی پاسخ‌ها مدل‌های بزرگ‌تری آموزش می‌دهند؛ در Colab از منوی `Runtime → Change runtime type` گزینهٔ **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.2860,), (0.3530,))])
full_train = datasets.FashionMNIST('data', train=True, download=True, transform=transform)
test_set = datasets.FashionMNIST('data', train=False, download=True, transform=transform)
train_set, val_set = random_split(full_train, [50_000, 10_000], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=512, num_workers=2)
test_loader = DataLoader(test_set, batch_size=512, num_workers=2)

CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
train_tf = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])
test_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])

def train_one_epoch(model, loader, criterion, optimizer, device, scheduler=None):
    """One pass over the training data. `scheduler` is stepped after every batch if given."""
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        total_loss += criterion(logits, y).item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


def fit(model, train_loader, val_loader, optimizer, epochs, device, scheduler=None, verbose=True):
    """Train for `epochs` epochs and return the history of losses and accuracies."""
    criterion = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(1, epochs + 1):
        tr = train_one_epoch(model, train_loader, criterion, optimizer, device, scheduler)
        va = evaluate(model, val_loader, criterion, device)
        for k, v in zip(history, (*tr, *va)):
            history[k].append(v)
        if verbose:
            print(f'epoch {epoch:2d} | train loss {tr[0]:.4f} acc {tr[1]:.4f} | val loss {va[0]:.4f} acc {va[1]:.4f}')
    return history


def conv_block(c_in, c_out, batchnorm=True, downsample='maxpool'):
    def bn(c):
        return [nn.BatchNorm2d(c)] if batchnorm else []
    layers = [nn.Conv2d(c_in, c_out, 3, padding=1), *bn(c_out), nn.ReLU()]
    if downsample == 'maxpool':
        layers += [nn.Conv2d(c_out, c_out, 3, padding=1), *bn(c_out), nn.ReLU(), nn.MaxPool2d(2)]
    else:                                   # the second convolution halves the resolution itself
        layers += [nn.Conv2d(c_out, c_out, 3, stride=2, padding=1), *bn(c_out), nn.ReLU()]
    return nn.Sequential(*layers)


class SimpleCNN(nn.Module):
    def __init__(self, downsample='maxpool'):
        super().__init__()
        self.features = nn.Sequential(conv_block(1, 32, downsample=downsample), conv_block(32, 64, downsample=downsample))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 10))

    def forward(self, x):
        return self.classifier(self.features(x))


class CifarCNN(nn.Module):
    def __init__(self, batchnorm=True):
        super().__init__()
        self.features = nn.Sequential(conv_block(3, 64, batchnorm), conv_block(64, 128, batchnorm), conv_block(128, 256, batchnorm))
        self.classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, 10))

    def forward(self, x):
        return self.classifier(self.features(x))


def n_params(m):
    return sum(p.numel() for p in m.parameters())

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

تابع `conv2d_naive` را طوری گسترش دهید که `stride` و `padding` را پشتیبانی کند و خروجی آن را با `F.conv2d` مقایسه کنید.

</div>

In [ ]:
def conv2d_naive(image, kernel, stride=1, padding=0):
    image = np.pad(image, padding)                               # zeros around the image
    kh, kw = kernel.shape
    H_out = (image.shape[0] - kh) // stride + 1
    W_out = (image.shape[1] - kw) // stride + 1
    out = np.zeros((H_out, W_out))
    for i in range(H_out):
        for j in range(W_out):
            r, c = i * stride, j * stride
            out[i, j] = np.sum(image[r:r + kh, c:c + kw] * kernel)
    return out


rng = np.random.default_rng(0)
img, ker = rng.normal(size=(9, 11)), rng.normal(size=(3, 3))
for stride, padding in [(1, 0), (1, 1), (2, 0), (2, 1), (3, 2)]:
    ours = conv2d_naive(img, ker, stride, padding)
    ref = F.conv2d(torch.tensor(img).view(1, 1, 9, 11), torch.tensor(ker).view(1, 1, 3, 3),
                   stride=stride, padding=padding).squeeze().numpy()
    print(f'stride={stride} padding={padding}: shape {ours.shape} | max difference {np.abs(ours - ref).max():.1e}')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

`np.pad` با مقدار پیش‌فرض صفر دقیقاً همان **Zero Padding** در PyTorch است. ابعاد خروجی از همان فرمول $\lfloor (H + 2p - k)/s \rfloor + 1$ به دست می‌آید و نتیجه تا دقت ممیز شناور با `F.conv2d` یکی است.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

ابعاد خروجی و تعداد پارامترهای هر لایه `CifarCNN` را دستی حساب کنید و با خروجی کد مقایسه کنید.

</div>

In [ ]:
def hand_count(c_in, c_out, k=3):
    conv = c_out * (c_in * k * k + 1)
    bn = 2 * c_out                                        # gamma and beta (running stats are buffers, not parameters)
    return conv, bn


h = 32
total = 0
print(f"{'layer':28} {'output shape':>16} {'params (hand)':>14}")
for c_in, c_out in [(3, 64), (64, 128), (128, 256)]:
    for name, (ci, co) in [('conv 3x3', (c_in, c_out)), ('conv 3x3', (c_out, c_out))]:
        conv, bn = hand_count(ci, co)
        total += conv + bn
        print(f'{name} {ci}->{co:<14} {str((co, h, h)):>16} {conv:>9,} + BN {bn}')
    h //= 2
    print(f'{"max pool 2x2":28} {str((c_out, h, h)):>16} {0:>14}')
fc = 256 * 10 + 10
total += fc
print(f'{"global avg pool + linear":28} {str((10,)):>16} {fc:>14,}')

model = CifarCNN()
print(f'\ntotal by hand: {total:,} | n_params(CifarCNN()): {n_params(model):,}')
x = torch.randn(1, 3, 32, 32)
for i, block in enumerate(model.features):
    x = block(x); print(f'after block {i + 1}: {tuple(x.shape)}')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

هر کانولوشن ۳×۳ با `padding=1` ابعاد مکانی را حفظ می‌کند و هر Max Pooling آن را نصف می‌کند: $32 \to 16 \to 8 \to 4$. سهم پارامترها به شدت در لایه‌های عمیق متمرکز است: کانولوشن ۲۵۶→۲۵۶ به‌تنهایی حدود ۵۹۰ هزار پارامتر دارد، چون تعداد پارامتر با **حاصل‌ضرب** کانال‌های ورودی و خروجی رشد می‌کند، نه با اندازهٔ تصویر. لایهٔ Linear آخر به لطف Global Average Pooling فقط ۲۵۷۰ پارامتر دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

در `SimpleCNN`، `MaxPool2d` را با کانولوشن با `stride=2` جایگزین کنید. تعداد پارامتر و دقت چه تغییری می‌کند؟

</div>

In [ ]:
EPOCHS = 10
for downsample in ('maxpool', 'stride'):
    torch.manual_seed(0)
    cnn = SimpleCNN(downsample).to(device)
    opt = torch.optim.AdamW(cnn.parameters(), lr=1e-3, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, epochs=EPOCHS, steps_per_epoch=len(train_loader))
    fit(cnn, train_loader, val_loader, opt, EPOCHS, device, scheduler=sched, verbose=False)
    test_acc = evaluate(cnn, test_loader, nn.CrossEntropyLoss(), device)[1]
    print(f'{downsample:8}: parameters {n_params(cnn):,} | test accuracy {test_acc:.4f}')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

تعداد پارامترها **دقیقاً یکسان** است: کانولوشن با `stride=2` همان وزن‌های کانولوشن معمولی را دارد و Max Pooling هم پارامتری نداشت. تفاوت در محاسبات است: کانولوشن گام‌دار خروجی را فقط در نیمی از مکان‌ها (در هر جهت) حساب می‌کند، پس آن لایه حدود ۴ برابر ارزان‌تر است. دقت دو روش معمولاً بسیار نزدیک است. در کانولوشن گام‌دار، شبکه **یاد می‌گیرد** چطور ابعاد را کم کند. ResNet و بیشتر معماری‌های مدرن از همین روش استفاده می‌کنند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

لایه‌های BatchNorm را از `CifarCNN` حذف کنید و منحنی‌های آموزش را با حالت قبل مقایسه کنید.

</div>

In [ ]:
cifar_train = datasets.CIFAR10('data', train=True, download=True, transform=train_tf)
cifar_test = datasets.CIFAR10('data', train=False, download=True, transform=test_tf)
CIFAR_CLASSES = cifar_train.classes
cifar_train_loader = DataLoader(cifar_train, batch_size=128, shuffle=True, num_workers=2)
cifar_test_loader = DataLoader(cifar_test, batch_size=512, num_workers=2)

CIFAR_EPOCHS = 15
cifar_histories, cifar_models = {}, {}
for bn in (True, False):
    torch.manual_seed(0)
    m = CifarCNN(batchnorm=bn).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=5e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, epochs=CIFAR_EPOCHS, steps_per_epoch=len(cifar_train_loader))
    name = 'with BatchNorm' if bn else 'without BatchNorm'
    cifar_histories[name] = fit(m, cifar_train_loader, cifar_test_loader, opt, CIFAR_EPOCHS, device, scheduler=sched, verbose=False)
    cifar_models[name] = m
    print(f'{name:18}: final test acc {cifar_histories[name]["val_acc"][-1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, h in cifar_histories.items():
    axes[0].plot(h['train_loss'], label=name); axes[1].plot(h['val_acc'], label=name)
axes[0].set_title('training loss'); axes[1].set_title('test accuracy')
for ax in axes:
    ax.set_xlabel('epoch'); ax.legend()
plt.show()

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

بدون BatchNorm، Loss در Epochهای اول بسیار کندتر پایین می‌آید. با `max_lr=3e-3` در OneCycle، آموزش حتی ممکن است در میانه ناپایدار شود (افزایش ناگهانی Loss) و دقت نهایی هم پایین‌تر است. BatchNorm توزیع ورودی هر لایه را در طول آموزش ثابت نگه می‌دارد و اجازه می‌دهد با نرخ یادگیری بزرگ‌تر سریع‌تر آموزش دهیم. برای آموزش بدون BatchNorm معمولاً باید نرخ یادگیری را کم کرد، که یعنی Epochهای بیشتر.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۵

Confusion Matrix مدل CIFAR-10 را رسم کنید. کدام کلاس‌ها بیشتر با هم اشتباه گرفته می‌شوند؟ آیا منطقی است؟

</div>

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


@torch.no_grad()
def predict_all(model, loader):
    model.eval()
    preds, labels = [], []
    for X, y in loader:
        preds.append(model(X.to(device)).argmax(1).cpu()); labels.append(y)
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()


preds, labels = predict_all(cifar_models['with BatchNorm'], cifar_test_loader)
cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(cm, display_labels=CIFAR_CLASSES).plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
pairs = sorted(((off[i, j] + off[j, i], CIFAR_CLASSES[i], CIFAR_CLASSES[j])
                for i in range(10) for j in range(i + 1, 10)), reverse=True)[:5]
for count, a, b in pairs:
    print(f'{a:>10} <-> {b:<10}: {count} mistakes')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

بیشترین اشتباه تقریباً همیشه بین **گربه و سگ** است: هر دو حیوان چهارپا و پشمالو در اندازه و پس‌زمینهٔ مشابه‌اند و در وضوح ۳۲×۳۲ جزئیات تمایزدهنده (شکل پوزه و گوش) چند پیکسل بیشتر نیست. جفت‌های بعدی هم معمولاً منطقی‌اند: خودرو و کامیون، هواپیما و کشتی (پس‌زمینهٔ آبی آسمان یا دریا)، و حیوانات چهارپا با هم (گوزن، اسب، گربه). یعنی خطاهای مدل بیشتر «انسانی» است تا تصادفی.

</div>